# 基金评价因子及基金评价体系

基于华泰金工研报完整复现

In [ ]:
import sys
sys.path.insert(0, r'C:\Users\chenh\.qclaw\workspace\fund_evaluation_system')

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

from config import setup_chinese_font, OUTPUT_DIR
from source.data_loader import get_fund_nav, get_market_index
from source.factor_calculator import calc_all_factors
from source.backtest_engine import calc_rank_ic, evaluate_factor_effectiveness
from source.fund_scorer import FundScorer
from source.plot import plot_ic_series, plot_factor_effectiveness

setup_chinese_font()
print('模块导入成功！')

## 1. 获取基金净值数据

In [ ]:
# 示例基金
fund_code = '110011'  # 易方达中小盘

end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=730)).strftime('%Y-%m-%d')

nav_df = get_fund_nav(fund_code, start_date, end_date)
print(f'净值数据: {len(nav_df)} 条')
nav_df.head()

## 2. 计算基金评价因子（31个）

In [ ]:
# 获取市场指数作为基准
market_index = get_market_index('000001', start_date, end_date)

# 计算31个因子
factors = calc_all_factors(
    nav_df['nav'],
    market_index['daily_return'] if len(market_index) > 0 else nav_df['daily_return'],
    window=252
)

print('\n计算因子数:', len(factors))
print('\n因子列表:')
for i, (name, value) in enumerate(factors.items(), 1):
    print(f'{i:2d}. {name:20s}: {value:.6f}' if not pd.isna(value) else f'{i:2d}. {name:20s}: N/A')

## 3. 因子分类

In [ ]:
factor_categories = {
    '收益获取能力': ['年化收益率'],
    '风险控制能力': ['波动率', '下行风险', '最大回撤', '回撤最大回补天数', 'VaR', 'beta'],
    '风险调整收益': ['夏普比率', '索提诺比率', '卡玛比率', '特雷诺比率'],
    '牛熊市表现': ['顺境收益率', '逆境收益率', '顺境战胜市场胜率', '逆境战胜市场胜率'],
    '选股能力': ['单因子模型alpha', 'T-M模型alpha', 'H-M模型alpha'],
    '择时能力': ['T-M模型择时', 'H-M模型择时'],
    '业绩持续性': ['Hurst指数'],
}

print('因子分类:')
for category, factor_list in factor_categories.items():
    print(f'\n{category}:')
    for f in factor_list:
        if f in factors.index:
            val = factors[f]
            print(f'  - {f}: {val:.4f}' if not pd.isna(val) else f'  - {f}: N/A')

## 4. 五维基金评分

In [ ]:
# 五维评分模型
scorer = FundScorer()

# 单基金评分展示
factor_df = pd.DataFrame([factors], index=[fund_code])
score_df = scorer.calculate_scores(factor_df)

print('五维评分结果:')
print(score_df)

## 5. 查看输出结果

In [ ]:
import os

output_files = os.listdir(OUTPUT_DIR)
print('输出文件:')
for f in output_files:
    print(f'  - {f}')